**Initial Codebase and Decision-Making**

The initial codebase for Function 5 ("Yield in a Chemical Reaction") was based on a **Bayesian Optimization (BO) framework** implemented in Python. This framework is particularly well-suited for optimizing expensive, black-box functions where direct gradient information is unavailable, and the goal is to maximize an output.

My decision to choose this codebase as a starting point was driven by:
* **Problem Alignment:** Function 5 involves optimizing a 4-dimensional black-box function to maximize yield. BO is ideal for such multi-dimensional black-box maximization problems.
* **Unimodal Nature:** The problem states the function is unimodal, meaning it has a single global maximum. While this simplifies the search compared to multi-modal functions, BO's ability to model the function's landscape and quantify uncertainty efficiently guides the search to this single peak.
* **Efficiency:** Evaluating the yield of a chemical process can be computationally expensive. BO aims to find the optimum with the fewest possible function evaluations.
* **Uncertainty Quantification:** Gaussian Processes (GPs), which underpin BO, provide both predictions and uncertainty estimates. This is valuable for intelligently guiding exploration, even for unimodal functions, by indicating where to sample next to refine the estimate of the peak.
* **Library Availability:** The `sklearn.gaussian_process` library offers a robust GP implementation, and `scikit-optimize` provides convenient tools for BO components, making the implementation accessible.

The initial data provided for Function 5 included 20 input-output pairs. Analysis showed the input `[0.22418902, 0.84648049, 0.87948418, 0.87851568]` resulted in the highest initial yield of `1088.8596181962705`. This immediately guided the initial search strategy towards exploiting the vicinity of this promising high-yield region.

### Code Modification

As the competition progressed, I made key modifications, primarily driven by a combination of insights from query results and structured learning from the program.

* **Initial Query Strategy Refinement (Week 1-2)**
    * **Modification:** After identifying `[0.22418902, 0.84648049, 0.87948418, 0.87851568]` as the highest initial yield point, the first query was set to be a slight adjustment from this point (`[0.23, 0.85, 0.88, 0.88]`).
    * **Reasoning:** Given the unimodal nature, exploiting the region around the highest observed point is an efficient strategy for finding the global maximum. Making small adjustments aimed to refine the solution in the immediate vicinity and efficiently gather new, useful data.
    * **Score Improvement:** This strategy proved effective, with the query `[0.23456, 0.852345, 0.882263, 0.882634]` yielding `1148.0546699002757`, which was an improvement over the `1088.8596181962705` initial best. This confirmed the value of local exploitation.

* **Refining Bayesian Optimization Parameters **
    * **Modification:** The `objective_function` was updated to include a placeholder unimodal function with a maximum at `[0.5, 0.5, 0.5, 0.5]`, simulating a more controlled environment for testing optimization (e.g., `1500 * np.exp(-np.sum((chemical_values - 0.5)**2) / 0.1) + np.random.normal(0, 50)`). `n_calls` for `gp_minimize` was increased (e.g., from an implicit lower value to `n_initial_points + 5`) to allow for more optimization steps beyond the initial points.
    * **Reasoning:** The placeholder function allowed for controlled experimentation with the optimizer's behavior. Adjusting `n_calls` ensures sufficient budget for the optimizer to explore and converge.
    * **Score Improvement:** This set the stage for controlled testing and demonstrated how Nelder-Mead could quickly converge to the theoretical maximum of the *placeholder function* (e.g., `999.9999954658434` near `[0.5, 0.5, 0.5, 0.5]`), proving the local refinement strategy.

* **Combining BO and Local Search**
    * **Modification:** Implemented a two-stage optimization: 1) An initial `gp_minimize` run (`result_bayesian`) for broader exploration and finding a good starting point. 2) A `Nelder-Mead` (`minimize` from `scipy.optimize`) refinement step, starting from the best point found by `gp_minimize` (`result_bayesian.x`).
    * **Reasoning:** Bayesian Optimization is effective for global exploration, while Nelder-Mead is a strong local optimizer for unimodal functions. Combining them exploits the strengths of both: BO finds the promising region, and Nelder-Mead fine-tunes the peak within that region.
    * **Score Improvement:** This hybrid approach significantly improved the precision of finding the optimum. The Nelder-Mead step consistently pushed the yield towards the theoretical maximum of the placeholder function (e.g., `999.9999954658434`), demonstrating very high efficiency in locating the peak.

* **Addressing Unexpected Low Yields (Mid-Competition Learning)**
    * **Observation:** A suggested query from the optimizer, close to a known high-yield point, resulted in a surprisingly low actual yield (29.366...).
    * **Reasoning (Latest Query & Program Material):** This phenomenon indicated that your function likely had an extremely sharp or localized peak. Even small deviations from the optimal input could lead to drastic drops in yield, suggesting the Gaussian Process model was either slightly mispredicting or deliberately exploring just off the peak due to its uncertainty. The most critical action was to UPDATE YOUR DATA with this new low-yield observation (`[0.487..., 0.493..., 0.529..., 0.504...]` with `29.366...` yield). This directly informed the optimizer about the steep drop-off, allowing its internal model to become more accurate and cautious in that specific region for future suggestions.
    * **Impact on Scores:** While this single observation was low, it was invaluable in guiding the optimizer. Continual iteration with updated data allowed the BO model to refine its understanding of the function landscape, ultimately contributing to finding higher peaks.

* **Addressing ConvergenceWarning (Mid-Competition Refinement)**
    * **Observation:** You received a `ConvergenceWarning` indicating that the optimal `length_scale` for your Gaussian Process kernel was found close to its upper bound.
    * **Reasoning (Program Material):** This warning, discussed in the context of Gaussian Processes, suggests that the true optimal `length_scale` might be higher than the default upper limit, potentially constraining the model's ability to best fit the data.
    * **Modification:** The recommendation was to increase the `length_scale_bounds` for the Matern kernel (e.g., to `(1e-5, 1e6)`), if it was being used. (Note: The provided code snippets show `RBF` kernel, not `Matern`, but the principle of adjusting bounds applies).
    * **Impact:** This change directly improved the accuracy of the underlying Gaussian Process model, leading to better predictions and potentially more effective next queries, thereby indirectly contributing to score improvements.

* **Acquisition Function (`acq_func`) Discussion**
    * **Consideration:** You enquired about using Upper Confidence Bound (UCB).
    * **Reasoning (Program Material):** Your existing `gp_minimize` setup defaults to Expected Improvement (EI) as the acquisition function. EI is generally effective for finding the global optimum. UCB, on the other hand, can be beneficial for broader exploration or when model certainty is low, with its kappa parameter controlling the exploration-exploitation balance. Notably, when minimizing the negative yield, using LCB (Lower Confidence Bound) with `kappa=1.96` is mathematically equivalent to using UCB for maximising the positive yield.
    * **Decision:** While there was no strict requirement to switch, the ability to choose between EI and UCB/LCB provided a tool for fine-tuning the balance if initial convergence became an issue.

**Final Result**

In the final weeks of the competition, the combined Bayesian Optimization and Nelder-Mead approach proved highly effective for Function 5.

* **Highest Yield Achieved:** Best input parameters found so far: [0.49997, 0.500016, 0.499966, 0.499997]
Highest observed yield so far: 1879.787779753955 The best yield from the loaded initial data was `1879.787779753955`.
* **Overall Improvement:** The strategy successfully navigated the 4-dimensional space and precisely located the maximum of the unimodal function, demonstrating robust convergence.

**Actions to Improve Code (If More Weeks Available):**

1.  **Transition to Actual Objective Function:** The most critical next step would be to replace the `dummy_objective_function` and placeholder yield calculations with the *actual code that evaluates the yield of the chemical process*. This is the real black-box component of the problem.
2.  **Robustness to Noise in Real Data:** The current placeholder function has controlled noise. Real chemical processes can be very noisy. Further refinement of the GP kernel (e.g., `Matern` kernels, adjusting `alpha` in `GaussianProcessRegressor`) might be needed to better handle varying noise levels if the actual process is noisier.
3.  **Dynamic Nelder-Mead Initialization/Iterations:** Explore whether the Nelder-Mead `max_iter` or `step_size` needs dynamic adjustment based on how close the Bayesian Optimization gets to the true optimum.
4.  **Batch Evaluations:** If multiple queries can be run in parallel (e.g., on different experimental setups in a lab), implement batch evaluations within the BO loop to accelerate data collection.
5.  **Uncertainty-Based Stopping Criteria:** Implement criteria to stop the optimization when the uncertainty (predicted standard deviation) around the predicted maximum becomes very low, indicating that further evaluations may not yield significant improvements.

**What I Would Do Differently if I Started Over Again:**

1.  **Formalize `objective_function` Early:** I would ensure that the `objective_function` (the one passed to `gp_minimize` and `minimize`) is designed from the very beginning to either call the actual chemical process or a highly faithful simulation of it.
2.  **Initial Sampling Strategy:** For the first few evaluations, I would employ a more systematic initial sampling strategy like Latin Hypercube Sampling (LHS) or Sobol sequences, rather than pure random or just taking initial provided points. This ensures a better initial coverage of the 4D space before the GP takes over.
3.  **Adaptive `n_calls` for `gp_minimize`:** Instead of a fixed `n_calls_bayesian`, I would explore making it adaptive or increasing it significantly (e.g., 50-100 initial BO iterations) to ensure broader exploration before switching to Nelder-Mead.
4.  **Visualization of Convergence:** Set up automated plots showing the best yield found versus the number of iterations. This would provide clear visual feedback on the convergence rate and indicate if the optimizer is stuck or progressing.
5.  **Handle Real-World Constraints:** If there were constraints on the chemical concentrations (e.g., sum of concentrations must be 1, or specific compounds must be within certain ranges), I would integrate these into the `space` definition and possibly the `minimize` call's `bounds` or `constraints` arguments from the outset.

In [ ]:
import numpy as np

initial_inputs = np.load('/content/initial_inputs.npy')
initial_outputs = np.load('/content/initial_outputs.npy')

print("Initial Inputs Shape:", initial_inputs.shape)
print("Initial Outputs Shape:", initial_outputs.shape)
print("Initial Inputs:\n", initial_inputs)
print("Initial Outputs:\n", initial_outputs)

Initial Inputs Shape: (20, 4)
Initial Outputs Shape: (20,)
Initial Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016

In [ ]:
import numpy as np

# Load the initial inputs and outputs from the .npy files
initial_inputs = np.load('/content/initial_inputs.npy')
initial_outputs = np.load('/content/initial_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(initial_outputs)
best_input = initial_inputs[best_index]

print("Input with highest yield:")
print(best_input)
print("Yield value:", initial_outputs[best_index])

Input with highest yield:
[0.22418902 0.84648049 0.87948418 0.87851568]
Yield value: 1088.8596181962705


In [ ]:
import numpy as np

# Load existing data
initial_inputs = np.load('/content/initial_inputs.npy')
initial_outputs = np.load('/content/initial_outputs.npy')

# New data point
new_input = np.array([0.23456, 0.852345, 0.882263, 0.882634])
new_output = np.float64(1148.0546699002757)

# Add new data point to existing data
updated_inputs = np.vstack([initial_inputs, new_input])
updated_outputs = np.append(initial_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")

New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
!pip install scikit-optimize==1.0.1

ERROR: Could not find a version that satisfies the requirement scikit-optimize==1.0.1 (from versions: 0.1, 0.2, 0.3, 0.4rc1, 0.4, 0.5, 0.5.1, 0.5.2, 0.7rc4, 0.7, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.8.dev0, 0.8.0, 0.8.1, 0.9rc1, 0.9.0, 0.10rc3, 0.10.0, 0.10.1, 0.10.2)
ERROR: No matching distribution found for scikit-optimize==1.0.1


In [ ]:
import numpy as np
from skopt import gp_minimize # This should work now as skopt should be installed
from skopt.space import Real
from skopt.utils import use_named_args

ModuleNotFoundError: No module named 'skopt'

In [ ]:
!pip install scikit-optimize==1.0.1
%load_ext autoreload
%autoreload 2
import numpy as np
from skopt import gp_minimize # This should work now as skopt should be installed
from skopt.space import Real
from skopt.utils import use_named_args

ERROR: Could not find a version that satisfies the requirement scikit-optimize==1.0.1 (from versions: 0.1, 0.2, 0.3, 0.4rc1, 0.4, 0.5, 0.5.1, 0.5.2, 0.7rc4, 0.7, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.8.dev0, 0.8.0, 0.8.1, 0.9rc1, 0.9.0, 0.10rc3, 0.10.0, 0.10.1, 0.10.2)
ERROR: No matching distribution found for scikit-optimize==1.0.1


ModuleNotFoundError: No module named 'skopt'

In [ ]:
!pip install scikit-optimize==1.0.1 # Added ! to ensure this command runs in the shell

ERROR: Could not find a version that satisfies the requirement scikit-optimize==1.0.1 (from versions: 0.1, 0.2, 0.3, 0.4rc1, 0.4, 0.5, 0.5.1, 0.5.2, 0.7rc4, 0.7, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.8.dev0, 0.8.0, 0.8.1, 0.9rc1, 0.9.0, 0.10rc3, 0.10.0, 0.10.1, 0.10.2)
ERROR: No matching distribution found for scikit-optimize==1.0.1


In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 2.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(updated_outputs)
best_input = updated_inputs[best_index]

# Define a small perturbation for exploration
perturbation_scale = 0.05  # Adjust as needed

# Generate a new data point near the best input
new_input = best_input + np.random.uniform(-perturbation_scale, perturbation_scale, size=best_input.shape)

# Ensure the new input is within the desired bounds (if any)
# ... (Add code to clip or adjust new_input values if necessary)

print("New data point to explore:")
print(new_input)

# You can now use this 'new_input' to evaluate the objective function
# (chemical process yield) and add the result to your data for further exploration.

New data point to explore:
[0.19871326 0.88113475 0.91530522 0.87964605]


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.198713, -0.881134, -0.915305, -0.879646])
new_output = np.float64(1451.135331029435)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")

New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[ 0.19144708  0.03819337  0.60741781  0.41458414]
 [ 0.75865295  0.53651774  0.65600038  0.36034155]
 [ 0.43834987  0.8043397   0.21024527  0.15129482]
 [ 0.70605083  0.53419196  0.26424335  0.48208755]
 [ 0.83647799  0.19360965  0.6638927   0.78564888]
 [ 0.68343225  0.11866264  0.82904591  0.56757661]
 [ 0.55362148  0.66734998  0.32380582  0.81486975]
 [ 0.35235627  0.32224153  0.11697937  0.47311252]
 [ 0.15378571  0.72938169  0.42259844  0.44307417]
 [ 0.46344227  0.63002451  0.10790646  0.9576439 ]
 [ 0.67749115  0.35850951  0.47959222  0.07288048]
 [ 0.58397341  0.14724265  0.34809746  0.42861465]
 [ 0.30688872  0.31687813  0.62263448  0.09539906]
 [ 0.51114177  0.817957    0.72871042  0.11235362]
 [ 0.43893338  0.77409176  0.37816709  0.93369621]
 [ 0.22418902  0.84648049  0.87948418  0.87851568]
 [ 0.72526172  0.47987049  0.08894684  0.75976022]
 [ 0.35548161  0.63961937  0.41761768  0.12260384]
 [ 0.11987923  0.86254031  0.64333133  0.84980383]
 [ 0.12688467 

In [ ]:
import numpy as np

# Load the data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the index of the row to remove
row_index = np.where((updated_inputs == [0.198713, -0.881134, -0.915305, -0.879646]).all(axis=1))[0]

# Remove the row if found
if row_index.size > 0:
    updated_inputs = np.delete(updated_inputs, row_index[0], axis=0)
    updated_outputs = np.delete(updated_outputs, row_index[0])

# Add the new row with positive values
new_input = np.array([0.198713, 0.881134, 0.915305, 0.879646])
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, 1451.135331029435)  # Assuming the same output

# Save the updated data
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("Row removed, replaced with positive values, and data saved.")

Row removed, replaced with positive values, and data saved.


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    # ... (Your code to evaluate the chemical process yield using params) ...
    # Example: Assume 'yield_calculator' is your function to calculate yield
    # yield_value = yield_calculator(params['chemical_1'], params['chemical_2'],
    #                                   params['chemical_3'], params['chemical_4'])

    # For now, let's use a simple placeholder function:
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    # ... (Your code to evaluate the chemical process yield using params) ...
    # Example: Assume 'yield_calculator' is your function to calculate yield
    # yield_value = yield_calculator(params['chemical_1'], params['chemical_2'],
    #                                   params['chemical_3'], params['chemical_4'])

    # For now, let's use a simple placeholder function:
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, assume you would run your chemical process here and get the 'yield_value'
        yield_value = 0 # Placeholder, replace with actual yield calculation
    return yield_value

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

# ... (Your existing code to define space, load data, and define objective_function) ...

# Define the number of calls to the objective function
n_calls = 20  # Adjust as needed

# Run the optimization
result = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls,
    random_state=42,  # For reproducibility (optional)
    # You can add other parameters like 'acq_func', 'n_initial_points', etc.
    # See skopt documentation for details
)

# Print the results
print("Best input parameters:", result.x)
print("Best objective function value:", result.fun)

Best input parameters: [0.7965429868602331, 0.18343478986616382, 0.7796910002727695, 0.5968501579464871]
Best objective function value: 0


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.198713, 0.881134, 0.915305, 0.879646])
new_output = np.float64(1451.135331029435)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)


In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.796542, 0.183434, 0.779691, 0.59685 ])
new_output = np.float64(136.4887611670581)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np

<module 'numpy' from '/usr/local/lib/python3.11/dist-packages/numpy/__init__.py'>

In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
!pip install scikit-optimize==1.0.1

ERROR: Could not find a version that satisfies the requirement scikit-optimize==1.0.1 (from versions: 0.1, 0.2, 0.3, 0.4rc1, 0.4, 0.5, 0.5.1, 0.5.2, 0.7rc4, 0.7, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.8.dev0, 0.8.0, 0.8.1, 0.9rc1, 0.9.0, 0.10rc3, 0.10.0, 0.10.1, 0.10.2)
ERROR: No matching distribution found for scikit-optimize==1.0.1


In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

In [ ]:
# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

In [ ]:
# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

In [ ]:
# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    # ... (Your code to evaluate the chemical process yield using params) ...
    # Example: Assume 'yield_calculator' is your function to calculate yield
    # yield_value = yield_calculator(params['chemical_1'], params['chemical_2'],
    #                                   params['chemical_3'], params['chemical_4'])

    # For now, let's use a simple placeholder function:
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, assume you would run your chemical process here and get the 'yield_value'
        yield_value = 0 # Placeholder, replace with actual yield calculation

    # Since we want to maximize yield, we return the negative of the yield value
    # (gp_minimize seeks to minimize the objective function)
    return -yield_value

In [ ]:
# Define the number of calls to the objective function
n_calls = 20  # Adjust as needed

# Run the optimization
result = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls,
    random_state=42,  # For reproducibility (optional)
    # You can add other parameters like 'acq_func', 'n_initial_points', etc.
    # See skopt documentation for details
)

# Print the results
print("Best input parameters:", result.x)
print("Best objective function value:", result.fun)

Best input parameters: [0.7965429868602331, 0.18343478986616382, 0.7796910002727695, 0.5968501579464871]
Best objective function value: 0


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(updated_outputs)
best_input = updated_inputs[best_index]

print("Input with highest yield:")
print(best_input)
print("Yield value:", updated_outputs[best_index])

Input with highest yield:
[0.23456  0.852345 0.882263 0.882634]
Yield value: 1148.0546699002757


In [ ]:
import numpy as np
from scipy.optimize import minimize

# 1. Analyze Initial Data
# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(updated_outputs)
best_initial_input = updated_inputs[best_index]


# 2. Exploit the Maximum (Neighborhood is implicitly defined in Nelder-Mead)


# 3. Refine the Search (Using Nelder-Mead)

# Define the objective function (to be minimized, so return negative yield)
def objective_function(inputs):
  # Check if this input combination has already been evaluated
  if any(np.equal(updated_inputs, inputs).all(1)):
    existing_output_index = np.where(np.equal(updated_inputs, inputs).all(1))[0][0]
    yield_value = updated_outputs[existing_output_index]
  else:
    # If not, assume you would run your chemical process here and get the 'yield_value'
    yield_value = 0  # Placeholder, replace with actual yield calculation

  return -yield_value  # Minimize negative yield to maximize yield

# Initial guess for optimization (best input from initial data)
initial_guess = best_initial_input

# Perform optimization using Nelder-Mead
result = minimize(objective_function, initial_guess, method='Nelder-Mead')

# Print results
print("Optimized input:", result.x)
print("Maximum yield:", -result.fun)  # Negate to get actual yield


Optimized input: [0.23456  0.852345 0.882263 0.882634]
Maximum yield: 1148.0546699002757


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(updated_outputs)
best_input = updated_inputs[best_index]

# Define a small perturbation for exploration
perturbation_scale = 0.05  # Adjust as needed

# Generate a new data point near the best input
new_input = best_input + np.random.uniform(-perturbation_scale, perturbation_scale, size=best_input.shape)

# Ensure the new input is within the desired bounds (if any)
# ... (Add code to clip or adjust new_input values if necessary)

print("New data point to explore:")
print(new_input)

# You can now use this 'new_input' to evaluate the objective function
# (chemical process yield) and add the result to your data for further exploration.


New data point to explore:
[0.25864021 0.82261994 0.90083155 0.87429934]


In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.25864 , 0.822619, 0.900831, 0.874299])
new_output = np.float64(1081.046033060803)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")

New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(updated_outputs)  # Index of the maximum output
best_input = updated_inputs[best_index]  # Corresponding input

print("Input with highest yield:")
print(best_input)
print("Yield value:", updated_outputs[best_index])

Input with highest yield:
[0.23456  0.852345 0.882263 0.882634]
Yield value: 1148.0546699002757


In [ ]:
import numpy as np
from scipy.optimize import minimize

# 1. Analyze Initial Data
# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(updated_outputs)
best_initial_input = updated_inputs[best_index]


# 2. Exploit the Maximum (Neighborhood is implicitly defined in Nelder-Mead)


# 3. Refine the Search (Using Nelder-Mead)

# Define the objective function (to be minimized, so return negative yield)
def objective_function(inputs):
  # Check if this input combination has already been evaluated
  if any(np.equal(updated_inputs, inputs).all(1)):
    existing_output_index = np.where(np.equal(updated_inputs, inputs).all(1))[0][0]
    yield_value = updated_outputs[existing_output_index]
  else:
    # If not, assume you would run your chemical process here and get the 'yield_value'
    yield_value = 0  # Placeholder, replace with actual yield calculation

  return -yield_value  # Minimize negative yield to maximize yield

# Initial guess for optimization (best input from initial data)
initial_guess = best_initial_input

# Perform optimization using Nelder-Mead
result = minimize(objective_function, initial_guess, method='Nelder-Mead')

# Print results
print("Optimized input:", result.x)
print("Maximum yield:", -result.fun)  # Negate to get actual yield

Optimized input: [0.23456  0.852345 0.882263 0.882634]
Maximum yield: 1148.0546699002757


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(updated_outputs)
best_input = updated_inputs[best_index]

# Define a small perturbation for exploration
perturbation_scale = 0.05  # Adjust as needed

# Generate a new data point near the best input
new_input = best_input + np.random.uniform(-perturbation_scale, perturbation_scale, size=best_input.shape)

# Ensure the new input is within the desired bounds (if any)
# ... (Add code to clip or adjust new_input values if necessary)

print("New data point to explore:")
print(new_input)

# You can now use this 'new_input' to evaluate the objective function
# (chemical process yield) and add the result to your data for further exploration.

New data point to explore:
[0.23274645 0.86102956 0.84871271 0.92900876]


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.232746, 0.861029, 0.848712, 0.929008])
new_output = np.float64(1279.8791940467468)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")

New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.198713, -0.881134, -0.915305, -0.879646])
new_output = np.float64(1451.135331029435)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")

New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[ 0.19144708  0.03819337  0.60741781  0.41458414]
 [ 0.75865295  0.53651774  0.65600038  0.36034155]
 [ 0.43834987  0.8043397   0.21024527  0.15129482]
 [ 0.70605083  0.53419196  0.26424335  0.48208755]
 [ 0.83647799  0.19360965  0.6638927   0.78564888]
 [ 0.68343225  0.11866264  0.82904591  0.56757661]
 [ 0.55362148  0.66734998  0.32380582  0.81486975]
 [ 0.35235627  0.32224153  0.11697937  0.47311252]
 [ 0.15378571  0.72938169  0.42259844  0.44307417]
 [ 0.46344227  0.63002451  0.10790646  0.9576439 ]
 [ 0.67749115  0.35850951  0.47959222  0.07288048]
 [ 0.58397341  0.14724265  0.34809746  0.42861465]
 [ 0.30688872  0.31687813  0.62263448  0.09539906]
 [ 0.51114177  0.817957    0.72871042  0.11235362]
 [ 0.43893338  0.77409176  0.37816709  0.93369621]
 [ 0.22418902  0.84648049  0.87948418  0.87851568]
 [ 0.72526172  0.47987049  0.08894684  0.75976022]
 [ 0.35548161  0.63961937  0.41761768  0.12260384]
 [ 0.11987923  0.86254031  0.64333133  0.84980383]
 [ 0.12688467 

In [ ]:
import numpy as np

# Load the data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the index of the row to remove
row_index = np.where((updated_inputs == [0.198713, -0.881134, -0.915305, -0.879646]).all(axis=1))[0]

# Remove the row if found
if row_index.size > 0:
    updated_inputs = np.delete(updated_inputs, row_index[0], axis=0)
    # Check if row_index[0] is within the bounds of updated_outputs
    if row_index[0] < updated_outputs.shape[0]:
        updated_outputs = np.delete(updated_outputs, row_index[0])
    else:
        print(f"Warning: row_index {row_index[0]} is out of bounds for updated_outputs. Skipping deletion.")


# Add the new row with positive values
new_input = np.array([0.198713, 0.881134, 0.915305, 0.879646])
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, 1451.135331029435)  # Assuming the same output

# Save the updated data
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("Row removed, replaced with positive values, and data saved.")


Row removed, replaced with positive values, and data saved.


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np

# Load the data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Define the input-output pairs to update
updates = [
    (np.array([0.23456, 0.852345, 0.882263, 0.882634]), np.float64(1148.0546699002757)),
    (np.array([0.198713, 0.881134, 0.915305, 0.879646]), np.float64(1451.135331029435)),
    (np.array([0.796542, 0.183434, 0.779691, 0.59685]), np.float64(136.4887611670581)),
    (np.array([0.25864, 0.822619, 0.900831, 0.874299]), np.float64(1081.046033060803)),
    (np.array([0.232746, 0.861029, 0.848712, 0.929008]), np.float64(1279.8791940467468)),
]

# Update the outputs
for input_to_update, output_value in updates:
    row_index = np.where((updated_inputs == input_to_update).all(axis=1))[0]
    if row_index.size > 0:
        updated_outputs[row_index[0]] = output_value

# Save the updated data
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("Outputs updated and data saved.")


Outputs updated and data saved.


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
!pip install scikit-optimize
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
from scipy.optimize import minimize

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    # ... (Your code to evaluate the chemical process yield using params) ...
    # Example: Assume 'yield_calculator' is your function to calculate yield
    # yield_value = yield_calculator(params['chemical_1'], params['chemical_2'],
    #                                   params['chemical_3'], params['chemical_4'])

    # For now, let's use a simple placeholder function:
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, assume you would run your chemical process here and get the 'yield_value'
        yield_value = 0 # Placeholder, replace with actual yield calculation

    # Since we want to maximize yield, we return the negative of the yield value
    # (gp_minimize seeks to minimize the objective function)
    return -yield_value

# 1. Bayesian Optimization (Initial Exploration)
n_calls_bayesian = 10  # Number of calls for Bayesian Optimization
result_bayesian = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls_bayesian,
    random_state=42,
)

# 2. Nelder-Mead Refinement (Local Exploitation)
initial_guess_nelder_mead = result_bayesian.x  # Best input from Bayesian Optimization
result_nelder_mead = minimize(
    objective_function,
    initial_guess_nelder_mead,
    method='Nelder-Mead',
)

# Print the results
print("Best input parameters (Bayesian):", result_bayesian.x)
print("Best objective function value (Bayesian):", result_bayesian.fun)
print("Optimized input parameters (Nelder-Mead):", result_nelder_mead.x)
print("Maximum yield (Nelder-Mead):", -result_nelder_mead.fun)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.3 MB/s eta 0:00:00
Best input parameters (Bayesian): [0.7965429868602331, 0.18343478986616382, 0.7796910002727695, 0.5968501579464871]
Best objective function value (Bayesian): 0
Optimized input parameters (Nelder-Mead): [0.79654299 0.18343479 0.779691   0.59685016]
Maximum yield (Nelder-Mead): -0.0


In [ ]:
!pip install scikit-optimize
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
from scipy.optimize import minimize

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if any(np.equal(updated_inputs, chemical_values).all(1)):
        # If it exists, perturb the input slightly
        perturbation_scale = 0.05
        while any(np.equal(updated_inputs, chemical_values).all(1)):
            chemical_values = chemical_values + np.random.uniform(-perturbation_scale, perturbation_scale, size=chemical_values.shape)
            chemical_values = np.clip(chemical_values, 0, 1)  # Keep values within bounds

        # Assign a placeholder yield value (you would replace this with your yield calculation)
        yield_value = 0  # Replace with actual yield calculation for the perturbed point
    else:
        # If not, assume you would run your chemical process here and get the 'yield_value'
        yield_value = 0 # Placeholder, replace with actual yield calculation

    return -yield_value # Minimize negative yield to maximize yield

# 1. Bayesian Optimization (Initial Exploration)
n_calls_bayesian = 30  # Number of calls for Bayesian Optimization
result_bayesian = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls_bayesian,
    random_state=42,
)

# 2. Nelder-Mead Refinement (Local Exploitation)
# (Code for Nelder-Mead is omitted for brevity, but you can add it back)

# Get the next query point from the Bayesian optimization result
next_query_point = result_bayesian.x

# Check if this query point exists in updated_inputs
while any(np.equal(updated_inputs, next_query_point).all(1)):
    # Perturb the query point slightly until it's unique
    perturbation_scale = 0.05
    next_query_point = next_query_point + np.random.uniform(-perturbation_scale, perturbation_scale, size=next_query_point.shape)
    next_query_point = np.clip(next_query_point, 0, 1)  # Keep values within bounds

print("Next query point:", next_query_point)

Next query point: [0.7965429868602331, 0.18343478986616382, 0.7796910002727695, 0.5968501579464871]


In [ ]:
!pip install scikit-optimize --upgrade # Upgrade scikit-optimize to latest version
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args, halton # halton should now import correctly
from scipy.optimize import minimize

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    # ... (Your code to evaluate the chemical process yield using params) ...
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, assume you would run your chemical process here and get the 'yield_value'
        yield_value = 0 # Placeholder, replace with actual yield calculation

    return -yield_value # Minimize negative yield to maximize yield

# 1. Bayesian Optimization (Initial Exploration)
n_calls_bayesian = 10  # Number of calls for Bayesian Optimization
result_bayesian = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls_bayesian,
    random_state=42,
)

# 2. Generate a unique next query point using Halton sequence
dim = len(space) # Dimensionality of the search space
seq = halton(dim) # Generate Halton sequence
next_query_point = seq.generate(1)[0] # Get the next point in the sequence

# Ensure the next query point is unique
while any(np.isclose(updated_inputs, next_query_point).all(1)):
    next_query_point = seq.generate(1)[0] # Get the next point in the sequence

print("Next query point:", next_query_point)

ImportError: cannot import name 'halton' from 'skopt.utils' (/usr/local/lib/python3.11/dist-packages/skopt/utils.py)

In [ ]:
!pip install scikit-optimize --upgrade # Upgrade scikit-optimize to latest version
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
# Import halton from skopt.sampler
from skopt.sampler import halton
from scipy.optimize import minimize

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    # ... (Your code to evaluate the chemical process yield using params) ...
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, assume you would run your chemical process here and get the 'yield_value'
        yield

In [ ]:
!pip install scikit-optimize --upgrade

In [ ]:
from skopt.utils import halton

ImportError: cannot import name 'halton' from 'skopt.utils' (/usr/local/lib/python3.11/dist-packages/skopt/utils.py)

In [ ]:
!pip install scikit-optimize --upgrade # Upgrade scikit-optimize to latest version
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
from skopt.sampler import Halton
from scipy.optimize import minimize

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
except FileNotFoundError:
    updated_inputs = np.empty((0, 4))
    updated_outputs = np.empty((0,))
    print("Warning: updated_inputs.npy or updated_outputs.npy not found, starting with empty data.")

# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                 params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if updated_inputs.size > 0 and any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
        return -yield_value  # Minimize the negative to maximize yield (if yield is positive)
    else:
        # If not, we need to simulate running the chemical process
        # Replace this with your actual function to evaluate the yield
        yield_value = -np.sum((chemical_values - 0.5)**2)  # Example: yield is higher closer to 0.5 for all chemicals
        return -yield_value

# Determine the number of initial points
n_initial_points = updated_inputs.shape[0] if updated_inputs.size > 0 else 0

# Set n_calls to be at least the number of initial points + 1 (for the next query)
n_calls = max(10, n_initial_points + 1) # Increased n_calls

# Minimize the objective function using gp_minimize
res = gp_minimize(objective_function, space, n_calls=n_calls, random_state=1,
                  x0=updated_inputs.tolist() if updated_inputs.size > 0 else None,
                  y0=[-o for o in updated_outputs.tolist()] if updated_outputs.size > 0 else None)

# Extract the next query (the point that gp_minimize suggests for the next evaluation)
next_query = res.x

print("Next Query:", next_query)

# --- Code to simulate querying the black box and updating data ---
# In a real scenario, you would now use 'next_query' to run your experiment
# and obtain a new 'yield_value'.

# For this example, we'll just simulate the output using the objective function
simulated_yield = -objective_function(chemical_1=next_query[0],
                                      chemical_2=next_query[1],
                                      chemical_3=next_query[2],
                                      chemical_4=next_query[3])

# Update the stored data with the new query and its result
updated_inputs = np.vstack([updated_inputs, next_query])
updated_outputs = np.append(updated_outputs, simulated_yield)

# Save the updated data (optional, but good practice for persistence)
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("\nUpdated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

ValueError: Expected `n_calls` >= 10, got 1

In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)


Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Get indices of sorted outputs in descending order
sorted_indices = np.argsort(updated_outputs)[::-1]

# Get the top 3 indices
top_3_indices = sorted_indices[:3]

# Get the corresponding inputs and outputs
best_3_inputs = updated_inputs[top_3_indices]
best_3_outputs = updated_outputs[top_3_indices]

# Print the results
print("Top 3 Inputs:")
for i, input_values in enumerate(best_3_inputs):
    print(f"Input {i+1}: {input_values}, Output: {best_3_outputs[i]}")

Top 3 Inputs:
Input 1: [0.198713 0.881134 0.915305 0.879646], Output: 1451.135331029435
Input 2: [0.232746 0.861029 0.848712 0.929008], Output: 1279.8791940467468
Input 3: [0.23456  0.852345 0.882263 0.882634], Output: 1148.0546699002757


In [ ]:
!pip install scikit-optimize==1.0.1
import numpy as np
from skopt import gp_minimize
from skopt.space import Real, Space # Import Space
from skopt.utils import use_named_args
from skopt.sampler import Halton
from scipy.optimize import minimize

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]
# space = Space(space)  # Remove this line - gp_minimize expects a list of dimensions, not a Space object

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
updated_inputs = updated_inputs[:min_len]
updated_outputs = updated_outputs[:min_len]

# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                 params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if updated_inputs.size > 0 and any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
        return -yield_value  # Minimize the negative to maximize yield (if yield is positive)
    else:
        # If not, we need to simulate running the chemical process
        # Replace this with your actual function to evaluate the yield
        yield_value = -np.sum((chemical_values - 0.5)**2)  # Example: yield is higher closer to 0.5 for all chemicals
        return -yield_value

# Determine the number of initial points
n_initial_points = updated_inputs.shape[0] if updated_inputs.size > 0 else 0

# Set n_calls to be at least the number of initial points + 1 (for the next query)
n_calls = max(10, n_initial_points + 1) # Increased n_calls

# Minimize the objective function using gp_minimize
res = gp_minimize(objective_function, space, n_calls=n_calls, random_state=1,
                  x0=updated_inputs.tolist() if updated_inputs.size > 0 else None,
                  y0=[-o for o in updated_outputs.tolist()] if updated_outputs.size > 0 else None)

# Extract the next query (the point that gp_minimize suggests for the next evaluation)
next_query = res.x

# Check if the query point is already in updated_inputs
while any(np.isclose(updated_inputs, next_query).all(1)):
    # If it is, generate a new point using Halton sequence
    halton_sampler = Halton(n_samples=1)  # Create a Halton sampler for one sample
    # Use space.dimensions instead of space in generate()
    next_query = space.inverse_transform(halton_sampler.generate(space.dimensions))[0] # Get a new point

print("Next Query:", next_query)

ERROR: Could not find a version that satisfies the requirement scikit-optimize==1.0.1 (from versions: 0.1, 0.2, 0.3, 0.4rc1, 0.4, 0.5, 0.5.1, 0.5.2, 0.7rc4, 0.7, 0.7.1, 0.7.2, 0.7.3, 0.7.4, 0.8.dev0, 0.8.0, 0.8.1, 0.9rc1, 0.9.0, 0.10rc3, 0.10.0, 0.10.1, 0.10.2)
ERROR: No matching distribution found for scikit-optimize==1.0.1


TypeError: Halton.__init__() got an unexpected keyword argument 'n_samples'

In [ ]:
import numpy as np
from scipy.optimize import minimize

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the input that resulted in the highest yield
best_index = np.argmax(updated_outputs)
best_input = updated_inputs[best_index]

# Define the objective function (to be minimized, so return negative yield)
def objective_function(inputs):
    # Check if this input combination has already been evaluated
    if any(np.equal(updated_inputs, inputs).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, inputs).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, assume yield is 0 (or you can use a surrogate model)
        yield_value = 0
    return -yield_value  # Minimize negative yield to maximize yield

# Perform optimization using Nelder-Mead, starting from the best input
result = minimize(objective_function, best_input, method='Nelder-Mead')

# Print results
print("Optimized input:", result.x)
print("Maximum yield:", -result.fun)  # Negate to get actual yield

Optimized input: [0.198713 0.881134 0.915305 0.879646]
Maximum yield: 1451.135331029435


In [ ]:
!pip install scikit-optimize --upgrade  # Make sure scikit-optimize is up-to-date
!pip install scikit-learn  # Install scikit-learn for KNN
import numpy as np
from skopt.space import Space, Real
from sklearn.neighbors import NearestNeighbors

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Define the search space using Real for continuous values
space = Space([
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
])

# Find the top 3 best inputs
num_best_points = 3
best_indices = np.argsort(updated_outputs)[-num_best_points:]
best_inputs = updated_inputs[best_indices]

# Create a KNN model
k = 5  # Number of neighbors to consider
knn_model = NearestNeighbors(n_neighbors=k)
knn_model.fit(updated_inputs)

# Generate new query points near the best inputs with perturbation
new_query_points = []
perturbation_scale = 0.05  # Adjust as needed
for best_input in best_inputs:
    # Find the k-nearest neighbors to the best input
    distances, indices = knn_model.kneighbors([best_input])

    # Select a random neighbor (excluding the best input itself)
    neighbor_index = np.random.choice(indices[0][1:])  # Exclude the best input (index 0)
    neighbor_point = updated_inputs[neighbor_index]

    # Add perturbation to the neighbor point to create a new query point
    new_query_point = neighbor_point + np.random.uniform(-perturbation_scale, perturbation_scale, size=neighbor_point.shape)

    # Clip the new query point to ensure it stays within the search space bounds
    new_query_point = np.clip(new_query_point, 0, 1)

    # Add the new query point to the list
    new_query_points.append(new_query_point)

# Print the generated query points
print("New query points:")
for i, query_point in enumerate(new_query_points):
    print(f"Query {i+1}: {query_point}")

# You can also store the query points in a NumPy array for later use:
new_query_points_array = np.array(new_query_points)
print("\nNew query points array:\n", new_query_points_array)

New query points:
Query 1: [0.27656097 0.8078608  0.87483511 0.82999441]
Query 2: [0.17427238 0.86957849 0.87120328 0.8505883 ]
Query 3: [0.27007812 0.80631834 0.84476811 0.84548823]

New query points array:
 [[0.27656097 0.8078608  0.87483511 0.82999441]
 [0.17427238 0.86957849 0.87120328 0.8505883 ]
 [0.27007812 0.80631834 0.84476811 0.84548823]]


In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.174272, 0.869578, 0.871203, 0.850588])
new_output = np.float64(1024.229073999556)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")


New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
!pip install scikit-optimize
!pip install scikit-learn  # Install scikit-learn for KNN if you plan to use it

import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
from scipy.optimize import minimize

# Define the search space for the four chemicals (assuming they are ratios between 0 and 1)
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data (if any) or start with empty data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
except FileNotFoundError:
    updated_inputs = np.empty((0, 4))
    updated_outputs = np.empty((0,))
    print("Warning: updated_inputs.npy or updated_outputs.npy not found, starting with empty data.")

# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
if updated_inputs.shape[0] != updated_outputs.shape[0]:
    print(f"Warning: Mismatch in number of samples between inputs ({updated_inputs.shape[0]}) and outputs ({updated_outputs.shape[0]}). Trimming data to {min_len} samples.")
    updated_inputs = updated_inputs[:min_len]
    updated_outputs = updated_outputs[:min_len]


# Define the objective function (this is your black-box chemical process)
# You will need to replace the placeholder logic with your actual yield calculation
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                 params['chemical_3'], params['chemical_4']])

    # In a real scenario, you would use chemical_values to run your experiment
    # and get the actual yield. For this example, we'll use a placeholder.

    # Placeholder: A simple unimodal function (replace with your process)
    # This example function has a maximum at [0.5, 0.5, 0.5, 0.5]
    yield_value = 1000 - np.sum((chemical_values - 0.5)**2) * 2000 # Example: higher yield closer to 0.5

    # We want to maximize yield, but gp_minimize minimizes, so return the negative yield
    return -yield_value

# Combine Bayesian Optimization and Nelder-Mead
# This strategy uses Bayesian Optimization for exploration and Nelder-Mead for local exploitation

# 1. Bayesian Optimization (Exploration)
# Adjust n_calls based on how much exploration you need.
# n_initial_points should be set to the number of pre-existing data points.
n_initial_points = updated_inputs.shape[0]
n_calls_bayesian = max(20, n_initial_points + 10) # Ensure enough calls beyond initial points

res_bayesian = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls_bayesian,
    random_state=42,  # for reproducibility
    x0=updated_inputs.tolist() if updated_inputs.size > 0 else None,
    y0=[-o for o in updated_outputs.tolist()] if updated_outputs.size > 0 else None # gp_minimize minimizes
)

# 2. Nelder-Mead Refinement (Local Exploitation)
# Use the best point from Bayesian Optimization as the starting point for Nelder-Mead
initial_guess_nelder_mead = res_bayesian.x

# Minimize the negative objective function using Nelder-Mead
result_nelder_mead = minimize(
    objective_function,
    initial_guess_nelder_mead,
    method='Nelder-Mead',
)

# Print the results
print("Best input parameters found by Bayesian Optimization:", res_bayesian.x)
print("Best negative yield found by Bayesian Optimization:", res_bayesian.fun) # This is negative yield
print("\nOptimized input parameters found by Nelder-Mead:", result_nelder_mead.x)
print("Maximum yield found by Nelder-Mead:", -result_nelder_mead.fun) # Negate to get actual yield

# Note: In a real scenario, after running this code, you would take the
# 'Optimized input parameters' from Nelder-Mead, physically run your
# chemical process with these parameters to get the actual yield, and then
# add this new data point to your updated_inputs and updated_outputs files
# for potential future optimization runs.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.1 MB/s eta 0:00:00
Best input parameters found by Bayesian Optimization: [0.198713, 0.881134, 0.915305, 0.879646]
Best negative yield found by Bayesian Optimization: -1451.135331029435

Optimized input parameters found by Nelder-Mead: [0.49997037 0.50001646 0.49996665 0.49999756]
Maximum yield found by Nelder-Mead: 999.9999954658434


In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.49997 , 0.500016, 0.499966, 0.499997])
new_output = np.float64(32.0102004591918)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")


New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
!pip install scikit-optimize
!pip install scikit-learn  # Install scikit-learn for KNN if you plan to use it

import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
from scipy.optimize import minimize

# Define the search space for the four chemicals (assuming they are ratios between 0 and 1)
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data (if any) or start with empty data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
    print("Loaded existing data.")
except FileNotFoundError:
    updated_inputs = np.empty((0, 4))
    updated_outputs = np.empty((0,))
    print("No existing data found. Starting fresh.")

# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
if updated_inputs.shape[0] != updated_outputs.shape[0]:
    print(f"Warning: Mismatch in number of samples between inputs ({updated_inputs.shape[0]}) and outputs ({updated_outputs.shape[0]}). Trimming data to {min_len} samples.")
    updated_inputs = updated_inputs[:min_len]
    updated_outputs = updated_outputs[:min_len]
elif updated_inputs.shape[0] == 0:
    print("No data loaded or found. Starting with empty datasets.")


# Define the objective function (this is your black-box chemical process)
# You will need to replace the placeholder logic with your actual yield calculation
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                 params['chemical_3'], params['chemical_4']])

    # In a real scenario, you would use chemical_values to run your experiment
    # and get the actual yield. For this example, we'll use a placeholder.

    # Check if this input combination has already been evaluated
    # Use np.isclose for floating point comparison to avoid issues with exact equality
    # The rtol and atol parameters can be adjusted based on the desired precision
    is_evaluated = False
    if updated_inputs.size > 0:
        # Check if chemical_values is close to any row in updated_inputs
        is_evaluated = np.any(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1))

    if is_evaluated:
        # Find the index using a similar comparison
        existing_output_index = np.where(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1))[0][0]
        yield_value = updated_outputs[existing_output_index]
        print(f"Input {chemical_values} already evaluated. Returning existing yield: {yield_value}")
        # We want to maximize yield, but gp_minimize minimizes, so return the negative yield
        return -yield_value
    else:
        # Placeholder: A simple unimodal function (replace with your process)
        # This example function has a maximum at [0.5, 0.5, 0.5, 0.5]
        deviation_from_center = chemical_values - 0.5
        # A simple function where smaller deviations mean higher yield (maximum at 0.5)
        # Add some noise to simulate real-world experiments
        noise = np.random.normal(0, 50) # Increased noise for a more realistic simulation
        simulated_yield = 1500 - np.sum(deviation_from_center**2) * 5000 + noise # Adjusted scale for clearer differences
        # Ensure yield is not negative
        yield_value = max(0, simulated_yield)

        # In a real scenario, you would perform your chemical reaction with 'chemical_values'
        # and measure the actual 'yield_value'.

        # After getting the yield, update your data storage
        # Ensure updated_inputs and updated_outputs are global to be modified within the function
        global updated_inputs, updated_outputs
        updated_inputs = np.vstack([updated_inputs, chemical_values])
        updated_outputs = np.append(updated_outputs, yield_value)

        # Save the updated data (optional, but good for persistence)
        np.save('/content/updated_inputs.npy', updated_inputs)
        np.save('/content/updated_outputs.npy', updated_outputs)

        print(f"Evaluated new input: {chemical_values}. Yield: {yield_value}")

        return -yield_value # Minimize the negative yield to maximize the actual yield

# Combine Bayesian Optimization and Nelder-Mead
# This strategy uses Bayesian Optimization for exploration and Nelder-Mead for local exploitation

# 1. Bayesian Optimization (Exploration)
# Adjust n_calls based on how much exploration you need.
# n_initial_points should be set to the number of pre-existing data points.
# skopt will perform n_initial_points random initial evaluations if x0/y0 are not provided or are None.
# If x0/y0 are provided, it uses them and performs (n_calls - len(x0)) additional steps.
n_initial_points_from_data = updated_inputs.shape[0]

# Set the total number of calls. If you have initial data, skopt uses it first.
# Add extra calls for optimization beyond the initial data.
n_calls_total = n_initial_points_from_data + 30 # Perform 30 additional optimization steps after using initial data

print(f"Starting Bayesian Optimization with {n_initial_points_from_data} initial points from data and {n_calls_total - n_initial_points_from_data} additional steps (total {n_calls_total} calls).")

# Prepare x0 and y0 for gp_minimize. y0 should be the negative of the yields.
initial_x0 = updated_inputs.tolist() if updated_inputs.size > 0 else None
initial_y0 = [-o for o in updated_outputs.tolist()] if updated_outputs.size > 0 else None

res_bayesian = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls_total,
    random_state=42,  # for reproducibility
    x0=initial_x0, # Use existing data as initial points
    y0=initial_y0  # Use existing results (negative) as initial objective values
)

# 2. Nelder-Mead Refinement (Local Exploitation)
# Use the best point from Bayesian Optimization as the starting point for Nelder-Mead
# Ensure the starting point is within the defined bounds [0, 1]
initial_guess_nelder_mead = np.clip(res_bayesian.x, 0, 1)


# Minimize the negative objective function using Nelder-Mead
# Provide bounds for Nelder-Mead
bounds = [(0, 1) for _ in range(len(space))]

result_nelder_mead = minimize(
    objective_function, # Note: Nelder-Mead also calls the objective function
    initial_guess_nelder_mead,
    method='Nelder-Mead',
    bounds=bounds # Pass bounds to the minimize function
)

# Print the results
print("\nBayesian Optimization Results:")
print("Best input parameters found:", res_bayesian.x)
print("Best negative yield found:", res_bayesian.fun) # This is negative yield
print("\nOptimized input parameters found by Nelder-Mead:", result_nelder_mead.x)
print("Maximum yield found by Nelder-Mead:", -result_nelder_mead.fun) # Negate to get actual yield

# Note: In a real scenario, after running this code, you would take the
# 'Optimized input parameters' from Nelder-Mead, physically run your
# chemical process with these parameters to get the actual yield, and then
# add this new data point to your updated_inputs and updated_outputs files
# for potential future optimization runs.

Loaded existing data.


SyntaxError: name 'updated_inputs' is used prior to global declaration (<ipython-input-9-57ba6f8ca4c5>, line 79)

In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Find the index of the highest output value
best_index = np.argmax(updated_outputs)

# Get the input values corresponding to the highest output
best_input = updated_inputs[best_index]

# Get the highest output value itself
highest_output_value = updated_outputs[best_index]

# Print the results
print("Input corresponding to the highest output:", best_input)
print("Highest output value:", highest_output_value)

Input corresponding to the highest output: [0.198713 0.881134 0.915305 0.879646]
Highest output value: 1451.135331029435


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Get the indices that would sort updated_outputs in descending order
# np.argsort returns the indices in ascending order, so we use [::-1] to reverse for descending
sorted_indices = np.argsort(updated_outputs)[::-1]

# Get the top 3 indices
# Ensure we don't ask for more indices than available data points
top_n = 3
top_n_indices = sorted_indices[:min(top_n, len(updated_outputs))]

# Get the corresponding inputs and outputs using the top indices
best_n_inputs = updated_inputs[top_n_indices]
best_n_outputs = updated_outputs[top_n_indices]

# Print the top N results
print(f"Top {top_n} Inputs and their corresponding Outputs:")
for i in range(len(top_n_indices)):
    print(f"Rank {i+1}: Input = {best_n_inputs[i]}, Output = {best_n_outputs[i]}")

Top 3 Inputs and their corresponding Outputs:
Rank 1: Input = [0.198713 0.881134 0.915305 0.879646], Output = 1451.135331029435
Rank 2: Input = [0.232746 0.861029 0.848712 0.929008], Output = 1279.8791940467468
Rank 3: Input = [0.23456  0.852345 0.882263 0.882634], Output = 1148.0546699002757


In [ ]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

# Define the search space for the four chemicals (assuming they are ratios between 0 and 1)
# This should match the space used for previous optimization runs
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load the updated data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
    print(f"Loaded existing data with {updated_inputs.shape[0]} points.")
except FileNotFoundError:
    print("Error: Data files not found. Please ensure updated_inputs.npy and updated_outputs.npy exist.")
    # Exit or handle the error appropriately if data is missing
    updated_inputs = np.empty((0, len(space)))
    updated_outputs = np.empty((0,))


# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
if updated_inputs.shape[0] != updated_outputs.shape[0]:
    print(f"Warning: Mismatch in number of samples between inputs ({updated_inputs.shape[0]}) and outputs ({updated_outputs.shape[0]}). Trimming data to {min_len} samples.")
    updated_inputs = updated_inputs[:min_len]
    updated_outputs = updated_outputs[:min_len]


# Define a placeholder objective function for gp_minimize
# This function is only used by gp_minimize internally to build its model
# It doesn't need to *actually* run the experiment, just know the structure
# and accept inputs in the defined space. The real evaluation happens
# when you use the 'next_query' point.
@use_named_args(space)
def dummy_objective_function(**params):
    # This function is a placeholder. In a real scenario, this is where
    # you would run your actual chemical process.
    # Since we are just asking for the *next* query point based on *existing* data,
    # this function's actual return value for a new point is not used
    # to determine the *suggestion*, but it needs to exist for gp_minimize.
    # For existing points, we return the stored negative yield.
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                 params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if updated_inputs.size > 0 and any(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1)):
        existing_output_index = np.where(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1))[0][0]
        yield_value = updated_outputs[existing_output_index]
        return -yield_value # Minimize the negative yield
    else:
         # For points not in the historical data (which gp_minimize will explore),
         # the actual return value here isn't critical for the *suggestion* itself
         # when running with n_calls=len(x0)+1, as skopt uses its model.
         # We'll return a neutral value or something that doesn't skew the model.
         # However, gp_minimize *does* evaluate its internal model at the suggestion point.
         # For simplicity and consistency, you could return a placeholder value
         # or the model's prediction, but since we're just getting the suggestion,
         # the exact value for a *new* point here doesn't impact the next query.
         # A return is required though. Let's return a value that won't confuse the optimizer
         # if it unexpectedly tries to evaluate a point not in x0.
         # Returning a value that corresponds to the mean of the negative outputs is a safe choice.
         # If updated_outputs is empty, return 0.
         if updated_outputs.size > 0:
             return -np.mean(updated_outputs)
         else:
             return 0 # Or some other neutral value if no data exists


# Use gp_minimize with the existing data to suggest the next point
# We set n_calls to be one more than the number of existing data points.
# This tells gp_minimize to use the existing data (x0, y0) and then
# perform one additional step (the next query) based on its model.
n_initial_points = updated_inputs.shape[0]
n_calls_for_suggestion = n_initial_points + 1 # Ask for one additional evaluation

if n_initial_points == 0:
    print("Cannot suggest next query without initial data. Please add some data points first.")
else:
    print(f"Running gp_minimize to suggest the next query based on {n_initial_points} points.")
    # Prepare y0 as the negative of the outputs for minimization
    initial_x0 = updated_inputs.tolist()
    initial_y0 = [-o for o in updated_outputs.tolist()]

    # Run gp_minimize. It will use x0 and y0 and then suggest the next point.
    res = gp_minimize(
        dummy_objective_function, # Use the dummy function; real evaluation is external
        space,
        n_calls=n_calls_for_suggestion,
        random_state=42, # Use the same random state for reproducibility of suggestions
        x0=initial_x0,
        y0=initial_y0,
        n_initial_points=0 # Explicitly tell skopt *not* to do random initial points if x0 is provided
                          # It will use the provided x0/y0 and then proceed with optimization steps.
    )

    # The suggested next query point is the last point evaluated by gp_minimize
    # which is stored in res.x, *but* when using x0 and y0, the *next* suggested
    # point is often not directly in res.x after n_calls = len(x0) + 1.
    # A more reliable way to get the next suggestion is to look at the point
    # that the acquisition function is maximized at *after* training the model
    # on the data (x0, y0). However, getting this directly requires more
    # advanced skopt usage or running gp_minimize with a sampler.

    # A simpler approach is to just run gp_minimize for a few *new* calls
    # starting from the existing data. This will give you the points skopt
    # would choose next. Let's run for a small number of additional steps.
    n_additional_steps = 5 # Suggest the next 5 points skopt would explore

    print(f"Running gp_minimize for {n_additional_steps} additional steps to get next queries.")

    res_next_steps = gp_minimize(
        dummy_objective_function, # Use the dummy function
        space,
        n_calls=n_initial_points + n_additional_steps, # Total calls = existing + new steps
        random_state=42, # Keep random state consistent
        x0=initial_x0,
        y0=initial_y0,
        n_initial_points=0 # Use provided data first
    )

    # The new points suggested are the ones added beyond the initial points.
    # These are the last 'n_additional_steps' points in res_next_steps.x_iters
    # Note: x_iters contains all evaluated points, including the initial ones.
    all_evaluated_points = res_next_steps.x_iters
    next_queries = all_evaluated_points[n_initial_points:] # Points after the initial data

    print("\nSuggested next queries to evaluate:")
    for i, query in enumerate(next_queries):
        print(f"Query {i+1}: {query}")

    # The very first point in next_queries (if any) is the most immediate suggestion.
    if next_queries:
        most_immediate_next_query = next_queries[0]
        print("\nThe most immediate next query suggested by the optimizer is:")
        print(most_immediate_next_query)
    else:
        print("No new queries were suggested (perhaps n_additional_steps was 0 or less).")

Loaded existing data with 28 points.
Running gp_minimize to suggest the next query based on 27 points.
Running gp_minimize for 5 additional steps to get next queries.

Suggested next queries to evaluate:
Query 1: [0.005672863641690417, 0.8494054826958845, 0.9781431770721497, 0.9053045719575865]
Query 2: [0.2690869486468645, 0.10711371743317244, 0.9365299013147225, 1.0]
Query 3: [0.2198987959247526, 1.0, 0.4573974651467733, 0.10050717558573685]
Query 4: [0.2291628548928086, 1.0, 0.07520642295420488, 1.0]
Query 5: [0.2259814987076474, 1.0, 1.0, 1.0]
Query 6: [0.21479873075985084, 0.8853009966171217, 0.8366834397133899, 0.8288530809428057]
Query 7: [0.3579893845853197, 0.5867627301518407, 0.9261099935592499, 0.9348099133834926]
Query 8: [0.32928159963964426, 0.9739576393444227, 0.8573435138541395, 0.9311622649712487]
Query 9: [0.2020515407828466, 0.589072317920319, 0.7258257921550388, 0.9067846112078132]
Query 10: [0.2196745215953181, 0.6191007722074291, 0.9077698878540104, 0.904822867674

In [ ]:
import numpy as np

# The user provided the updated_outputs values directly
updated_outputs = np.array([
    6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
    2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
    8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
    6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
    2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00,
    1.14805467e+03, 1.45113533e+03, 1.36488761e+02, 1.08104603e+03,
    1.27987919e+03, 1.02422907e+03, 3.20102005e+01
])

# Find the highest output value using np.max()
highest_output = np.max(updated_outputs)

# Print the result
print("The highest output value is:", highest_output)

The highest output value is: 1451.13533


In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.005672, 0.849405, 0.978143, 0.905304])
new_output = np.float64(1879.7877797539552)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")

New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Get indices of sorted outputs in descending order
sorted_indices = np.argsort(updated_outputs)[::-1]

# Get the top 4 indices
top_4_indices = sorted_indices[:4]

# Get the corresponding inputs and outputs
best_4_inputs = updated_inputs[top_4_indices]
best_4_outputs = updated_outputs[top_4_indices]

# Print the results
print("Top 4 Inputs and corresponding Yields:")
for i, input_values in enumerate(best_4_inputs):
    print(f"Input {i+1}: {input_values}, Yield: {best_4_outputs[i]}")

Top 4 Inputs and corresponding Yields:
Input 1: [0.49997  0.500016 0.499966 0.499997], Yield: 1879.7877797539552
Input 2: [0.198713 0.881134 0.915305 0.879646], Yield: 1451.135331029435
Input 3: [0.232746 0.861029 0.848712 0.929008], Yield: 1279.8791940467468
Input 4: [0.23456  0.852345 0.882263 0.882634], Yield: 1148.0546699002757


In [ ]:
!pip install scikit-optimize
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
from skopt.sampler import Halton
from scipy.optimize import minimize

# Define the search space for the four chemicals
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
except FileNotFoundError:
    updated_inputs = np.empty((0, 4))
    updated_outputs = np.empty((0,))
    print("Warning: updated_inputs.npy or updated_outputs.npy not found, starting with empty data.")

# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
updated_inputs = updated_inputs[:min_len]
updated_outputs = updated_outputs[:min_len]


# Define the objective function (to be replaced with your actual chemical process)
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                 params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if updated_inputs.size > 0 and any(np.equal(updated_inputs, chemical_values).all(1)):
        existing_output_index = np.where(np.equal(updated_inputs, chemical_values).all(1))[0][0]
        yield_value = updated_outputs[existing_output_index]
        return -yield_value  # Minimize the negative to maximize yield (if yield is positive)
    else:
        # If not, we need to simulate running the chemical process
        # Replace this with your actual function to evaluate the yield
        yield_value = -np.sum((chemical_values - 0.5)**2)  # Example: yield is higher closer to 0.5 for all chemicals
        return -yield_value


# Determine the number of initial points
n_initial_points = updated_inputs.shape[0] if updated_inputs.size > 0 else 0

# Set n_calls to be at least the number of initial points + 1 (for the next query)
n_calls = max(10, n_initial_points + 1) # Increased n_calls

# Minimize the objective function using gp_minimize
res = gp_minimize(objective_function, space, n_calls=n_calls, random_state=1,
                  x0=updated_inputs.tolist() if updated_inputs.size > 0 else None,
                  y0=[-o for o in updated_outputs.tolist()] if updated_outputs.size > 0 else None)

# Extract the next query (the point that gp_minimize suggests for the next evaluation)
next_query = res.x

print("Next Query:", next_query)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.8 MB/s eta 0:00:00
Next Query: [0.49997, 0.500016, 0.499966, 0.499997]


In [ ]:
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

# Define the search space for the four chemicals (assuming they are ratios between 0 and 1)
# This should match the space used for previous optimization runs
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load the updated data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
    print(f"Loaded existing data with {updated_inputs.shape[0]} points.")
except FileNotFoundError:
    print("Error: Data files not found. Please ensure updated_inputs.npy and updated_outputs.npy exist.")
    # Exit or handle the error appropriately if data is missing
    updated_inputs = np.empty((0, len(space)))
    updated_outputs = np.empty((0,))

# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
if updated_inputs.shape[0] != updated_outputs.shape[0]:
    print(f"Warning: Mismatch in number of samples between inputs ({updated_inputs.shape[0]}) and outputs ({updated_outputs.shape[0]}). Trimming data to {min_len} samples.")
    updated_inputs = updated_inputs[:min_len]
    updated_outputs = updated_outputs[:min_len]

# Define a placeholder objective function for gp_minimize
# This function is only used by gp_minimize internally to build its model
# It doesn't need to *actually* run the experiment, just know the structure
# and accept inputs in the defined space. The real evaluation happens
# when you use the 'next_query' point.
@use_named_args(space)
def dummy_objective_function(**params):
    # This function is a placeholder. In a real scenario, this is where
    # you would run your actual chemical process.
    # Since we are just asking for the *next* query point based on *existing* data,
    # this function's actual return value for a new point is not used
    # to determine the *suggestion*, but it needs to exist for gp_minimize.
    # For existing points, we return the stored negative yield.
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                 params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    if updated_inputs.size > 0 and any(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1)):
        existing_output_index = np.where(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1))[0][0]
        yield_value = updated_outputs[existing_output_index]
        return -yield_value # Minimize the negative yield
    else:
         # For points not in the historical data (which gp_minimize will explore),
         # the actual return value here isn't critical for the *suggestion* itself
         # when running with n_calls=len(x0)+1, as skopt uses its model.
         # We'll return a neutral value or something that doesn't skew the model.
         # However, gp_minimize *does* evaluate its internal model at the suggestion point.
         # For simplicity and consistency, you could return a placeholder value
         # or the model's prediction, but since we're just getting the suggestion,
         # the exact value for a *new* point here doesn't impact the next query.
         # A return is required though. Let's return a value that won't confuse the optimizer
         # if it unexpectedly tries to evaluate a point not in x0.
         # Returning a value that corresponds to the mean of the negative outputs is a safe choice.
         # If updated_outputs is empty, return 0.
         if updated_outputs.size > 0:
             return -np.mean(updated_outputs)
         else:
             return 0 # Or some other neutral value if no data exists

# Use gp_minimize with the existing data to suggest the next point
# We set n_calls to be one more than the number of existing data points.
# This tells gp_minimize to use the existing data (x0, y0) and then
# perform one additional step (the next query) based on its model.
n_initial_points = updated_inputs.shape[0]
n_calls_for_suggestion = n_initial_points + 1 # Ask for one additional evaluation

if n_initial_points == 0:
    print("Cannot suggest next query without initial data. Please add some data points first.")
else:
    print(f"Running gp_minimize to suggest the next query based on {n_initial_points} points.")
    # Prepare y0 as the negative of the outputs for minimization
    initial_x0 = updated_inputs.tolist()
    initial_y0 = [-o for o in updated_outputs.tolist()]

    # Run gp_minimize. It will use x0 and y0 and then suggest the next point.
    res = gp_minimize(
        dummy_objective_function, # Use the dummy function; real evaluation is external
        space,
        n_calls=n_calls_for_suggestion,
        random_state=42, # Use the same random state for reproducibility of suggestions
        x0=initial_x0,
        y0=initial_y0,
        n_initial_points=0 # Explicitly tell skopt *not* to do random initial points if x0 is provided
                          # It will use the provided x0/y0 and then proceed with optimization steps.
    )

    # The suggested next query point is the last point evaluated by gp_minimize
    # which is stored in res.x, *but* when using x0 and y0, the *next* suggested
    # point is often not directly in res.x after n_calls = len(x0) + 1.
    # A more reliable way to get the next suggestion is to look at the point
    # that the acquisition function is maximized at *after* training the model
    # on the data (x0, y0). However, getting this directly requires more
    # advanced skopt usage or running gp_minimize with a sampler.

    # A simpler approach is to just run gp_minimize for a few *new* calls
    # starting from the existing data. This will give you the points skopt
    # would choose next. Let's run for a small number of additional steps.
    n_additional_steps = 5 # Suggest the next 5 points skopt would explore

    print(f"Running gp_minimize for {n_additional_steps} additional steps to get next queries.")

    res_next_steps = gp_minimize(
        dummy_objective_function, # Use the dummy function
        space,
        n_calls=n_initial_points + n_additional_steps, # Total calls = existing + new steps
        random_state=42, # Keep random state consistent
        x0=initial_x0,
        y0=initial_y0,
        n_initial_points=0 # Use provided data first
    )

    # The new points suggested are the ones added beyond the initial points.
    # These are the last 'n_additional_steps' points in res_next_steps.x_iters
    # Note: x_iters contains all evaluated points, including the initial ones.
    all_evaluated_points = res_next_steps.x_iters
    next_queries = all_evaluated_points[n_initial_points:] # Points after the initial data

    print("\nSuggested next queries to evaluate:")
    for i, query in enumerate(next_queries):
        print(f"Query {i+1}: {query}")

    # The very first point in next_queries (if any) is the most immediate suggestion.
    if next_queries:
        most_immediate_next_query = next_queries[0]
        print("\nThe most immediate next query suggested by the optimizer is:")
        print(most_immediate_next_query)
    else:
        print("No new queries were suggested (perhaps n_additional_steps was 0 or less).")




Loaded existing data with 29 points.
Running gp_minimize to suggest the next query based on 28 points.
Running gp_minimize for 5 additional steps to get next queries.

Suggested next queries to evaluate:
Query 1: [0.4872608115889606, 0.493727632774541, 0.5294452778139485, 0.504125500090696]
Query 2: [0.22745609151914964, 1.0, 0.1138784469456627, 0.1850241192072897]
Query 3: [0.27854305325122203, 0.898993550596839, 0.4305340237770857, 0.8557653163690653]
Query 4: [0.2247981539796423, 0.920426330565249, 1.0, 0.7185945512300908]
Query 5: [0.09026025705628558, 0.8845443202274357, 0.9404766823612669, 0.9066563550116763]
Query 6: [0.44007190892773107, 0.6401016540209448, 0.49247828309045466, 0.49895648245223506]
Query 7: [0.22435290687600798, 0.9824962994983348, 0.9317573230503007, 0.965283033117073]
Query 8: [0.2307722199977639, 0.8486812003827249, 1.0, 0.9420431934510786]
Query 9: [0.7768672559018611, 0.49331144243537606, 0.49404900675827834, 0.49916855774753666]
Query 10: [0.5030166881205

In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.4872608115889606, 0.493727632774541, 0.5294452778139485, 0.504125500090696])
new_output = np.float64(29.36665156358281)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")

New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
# Cell 1: Install scikit-optimize
!pip install scikit-optimize

# Cell 2: Import modules and run code
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
# Import Halton from skopt.sampler as it's used later in the notebook
from skopt.sampler import Halton
from scipy.optimize import minimize


# Define the search space for the four chemicals (assuming concentrations between 0 and 1)
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
except FileNotFoundError:
    updated_inputs = np.empty((0, 4))
    updated_outputs = np.empty((0,))
    print("Warning: updated_inputs.npy or updated_outputs.npy not found, starting with empty data.")

# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
if updated_inputs.shape[0] != updated_outputs.shape[0]:
    print(f"Warning: Mismatch in number of samples between inputs ({updated_inputs.shape[0]}) and outputs ({updated_outputs.shape[0]}). Trimming data to {min_len} samples.")
updated_inputs = updated_inputs[:min_len]
updated_outputs = updated_outputs[:min_len]


# Define the objective function (replace this with your actual chemical process simulation or experiment)
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    # Using np.isclose for floating point comparison
    if updated_inputs.size > 0 and any(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1)):
        existing_output_index = np.where(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, you would run your chemical process here and get the 'yield_value'.
        # For demonstration, let's use a placeholder function that simulates a unimodal yield.
        # This example function has a maximum yield at (0.5, 0.5, 0.5, 0.5).
        yield_value = 1500 * np.exp(-np.sum((chemical_values - 0.5)**2) / 0.1) + np.random.normal(0, 50) # Added noise

        # In a real scenario, you would NOT update your stored data here.
        # The data update should happen *after* you obtain the actual yield
        # from running the experiment with the suggested 'next_query'.
        # This objective function is only used by gp_minimize internally.


    # Since gp_minimize minimizes, we return the negative of the yield to maximize it.
    return -yield_value

# Determine the number of initial points from loaded data
n_initial_points = updated_inputs.shape[0] # Number of existing data points

# Set the total number of calls for gp_minimize
# This should be greater than n_initial_points to allow the optimizer to explore new points.
# For just getting the 'next' suggestion based on current data, you might run
# with n_calls = n_initial_points + 1, but running for a few more steps
# is a common way to see the sequence of suggestions.
n_calls = n_initial_points + 5 # Add 5 more calls for optimization

# Run the optimization using gp_minimize
# Provide the existing data using x0 and y0
res = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls,
    random_state=42,  # For reproducibility
    x0=updated_inputs.tolist() if updated_inputs.size > 0 else None, # Provide existing inputs
    y0=[-o for o in updated_outputs.tolist()] if updated_outputs.size > 0 else None, # Provide negative of existing outputs
    n_initial_points=0 # Tell skopt not to generate random points if x0 is provided
)

# The 'res' object contains the results of the optimization.
# res.x is the best input found among all evaluated points
# res.fun is the minimum objective function value (negative of the maximum yield)

print("Best input parameters found so far:", res.x)
print("Highest observed yield so far:", -res.fun) # Negate to get the actual maximum yield

# To get the next recommended point to test in your actual chemical process:
# These are the points evaluated by gp_minimize beyond the initial points provided in x0.
# They are stored in res.x_iters after the first n_initial_points.
if n_initial_points < len(res.x_iters):
    next_queries = res.x_iters[n_initial_points:]
    print("\nSuggested next queries to evaluate in your experiment:")
    for i, query in enumerate(next_queries):
        print(f"Suggestion {i+1}: {query}")

    # The very first point suggested is often the most relevant for the next step.
    most_immediate_next_query = next_queries[0]
    print("\nThe most immediate next query suggested by the optimizer is:")
    print(most_immediate_next_query)
else:
    print("\nNo new queries were suggested by gp_minimize in this run (n_calls was not greater than n_initial_points).")
    if updated_inputs.size > 0:
        # You could manually sample a point here if needed, e.g., using a sampler
        # For example, to get one random point within the space:
        # from skopt.sampler import Random
        # sampler = Random()
        # next_query_manual = sampler.generate(space, 1)[0]
        # print(f"Manual suggestion (random): {next_query_manual}")
        pass # Or handle the case where no new suggestions were made
    else:
         print("Cannot suggest next query without initial data.")

# --- After getting 'most_immediate_next_query', you would perform the real experiment ---
# You would run your chemical process with 'most_immediate_next_query' as input.
# Get the 'actual_yield_from_experiment'.
# Then, append this new input and its actual yield to your updated_inputs and updated_outputs arrays.
# Save the updated arrays to the .npy files.
# Repeat the process (load data, run gp_minimize for suggestions) in the next iteration.

# Example of how you would append the new data after your experiment:
# actual_yield_from_experiment = ... # Get this value from your real process/simulation
# updated_inputs = np.vstack([updated_inputs, most_immediate_next_query])
# updated_outputs = np.append(updated_outputs, actual_yield_from_experiment)
# np.save('/content/updated_inputs.npy', updated_inputs)
# np.save('/content/updated_outputs.npy', updated_outputs)
# print("\nAdded new data point from experiment.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.2 MB/s eta 0:00:00
Best input parameters found so far: [0.49997, 0.500016, 0.499966, 0.499997]
Highest observed yield so far: 1879.7877797539552

Suggested next queries to evaluate in your experiment:
Suggestion 1: [0.4940508552468735, 0.3375058324563514, 0.40991469615839876, 0.5144594214789104]
Suggestion 2: [0.5044928519730927, 0.4833442396298114, 0.7492726627826279, 0.41598688173161563]
Suggestion 3: [0.4653288951493909, 0.8296108978171678, 0.973970300176766, 0.5907728934464052]
Suggestion 4: [0.4979352248575434, 0.512661324528299, 0.46021978116530887, 0.5866916467168509]
Suggestion 5: [0.5106845765426401, 0.4587829343514061, 0.4780264546385208, 0.009625048808925145]
Suggestion 6: [0.5089192479890606, 0.9211439304047033, 0.5399370469965472, 0.5285303598260226]
Suggestion 7: [0.5019670049871175, 0.5232883259325772, 0.4618254137587117, 0.46215719980902054]
Suggestion 8: [0.4894987332832984, 0.5144187663230052, 0.529390086536

In [ ]:
import numpy as np

# Load existing data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# New data point
new_input = np.array([0.49405 , 0.337505, 0.409914, 0.514459])
new_output = np.float64(61.068009435706074)

# Add new data point to existing data
updated_inputs = np.vstack([updated_inputs, new_input])
updated_outputs = np.append(updated_outputs, new_output)

# Save updated data to files
np.save('/content/updated_inputs.npy', updated_inputs)
np.save('/content/updated_outputs.npy', updated_outputs)

print("New data point added and saved to updated_inputs.npy and updated_outputs.npy")


New data point added and saved to updated_inputs.npy and updated_outputs.npy


In [ ]:
import numpy as np

# Load the updated data
updated_inputs = np.load('/content/updated_inputs.npy')
updated_outputs = np.load('/content/updated_outputs.npy')

# Print the updated data
print("Updated Inputs:\n", updated_inputs)
print("Updated Outputs:\n", updated_outputs)

Updated Inputs:
 [[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.23456    0.852345   0.882263   0.882634

In [ ]:
# Cell 1: Install scikit-optimize
!pip install scikit-optimize

# Cell 2: Import modules and run code
import numpy as np
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
# Import Halton from skopt.sampler as it's used later in the notebook
from skopt.sampler import Halton
from scipy.optimize import minimize


# Define the search space for the four chemicals (assuming concentrations between 0 and 1)
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
except FileNotFoundError:
    updated_inputs = np.empty((0, 4))
    updated_outputs = np.empty((0,))
    print("Warning: updated_inputs.npy or updated_outputs.npy not found, starting with empty data.")

# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
if updated_inputs.shape[0] != updated_outputs.shape[0]:
    print(f"Warning: Mismatch in number of samples between inputs ({updated_inputs.shape[0]}) and outputs ({updated_outputs.shape[0]}). Trimming data to {min_len} samples.")
updated_inputs = updated_inputs[:min_len]
updated_outputs = updated_outputs[:min_len]


# Define the objective function (replace this with your actual chemical process simulation or experiment)
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    # Using np.isclose for floating point comparison
    if updated_inputs.size > 0 and any(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1)):
        existing_output_index = np.where(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, you would run your chemical process here and get the 'yield_value'.
        # For demonstration, let's use a placeholder function that simulates a unimodal yield.
        # This example function has a maximum yield at (0.5, 0.5, 0.5, 0.5).
        # In a real scenario, this function would call your simulation or interact with your experiment.
        # yield_value = your_chemical_process_function(chemical_values)
        yield_value = 1500 * np.exp(-np.sum((chemical_values - 0.5)**2) / 0.1) + np.random.normal(0, 50) # Added noise

        # In a real scenario, you would NOT update your stored data here.
        # The data update should happen *after* you obtain the actual yield
        # from running the experiment with the suggested 'next_query'.
        # This objective function is only used by gp_minimize internally.


    # Since gp_minimize minimizes, we return the negative of the yield to maximize it.
    return -yield_value

# Determine the number of initial points from loaded data
n_initial_points = updated_inputs.shape[0] # Number of existing data points

# Set the total number of calls for gp_minimize
# This should be greater than n_initial_points to allow the optimizer to explore new points.
# For just getting the 'next' suggestion based on current data, you might run
# with n_calls = n_initial_points + 1, but running for a few more steps
# is a common way to see the sequence of suggestions.
n_calls = n_initial_points + 5 # Add 5 more calls for optimization

# Run the optimization using gp_minimize
# Provide the existing data using x0 and y0
res = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls,
    random_state=42,  # For reproducibility
    x0=updated_inputs.tolist() if updated_inputs.size > 0 else None, # Provide existing inputs
    y0=[-o for o in updated_outputs.tolist()] if updated_outputs.size > 0 else None, # Provide negative of existing outputs
    n_initial_points=0 # Tell skopt not to generate random points if x0 is provided
)

# The 'res' object contains the results of the optimization.
# res.x is the best input found among all evaluated points
# res.fun is the minimum objective function value (negative of the maximum yield)

print("Best input parameters found so far:", res.x)
print("Highest observed yield so far:", -res.fun) # Negate to get the actual maximum yield

# To get the next recommended point to test in your actual chemical process:
# These are the points evaluated by gp_minimize beyond the initial points provided in x0.
# They are stored in res.x_iters after the first n_initial_points.
if n_initial_points < len(res.x_iters):
    # Take the last suggested point as the next query if you are only doing one experiment at a time
    # Or take the first point after the initial ones if you want the immediate next suggestion from the sequence
    # Here, we'll take the first point suggested after the initial ones.
    next_query = res.x_iters[n_initial_points]
    print("\nSuggested next query to evaluate in your experiment:")
    print(next_query)

else:
    print("\nNo new queries were suggested by gp_minimize in this run (n_calls was not greater than n_initial_points).")
    print("Consider increasing n_calls or check your data loading.")
    # In this case, you might want to manually generate a point to explore
    # For example, using a sampler like Random or Halton
    # from skopt.sampler import Random
    # sampler = Random(random_state=42)
    # next_query_manual = sampler.generate(space, 1)[0]
    # print(f"Manual suggestion (random): {next_query_manual}")


# --- After getting 'next_query', you would perform the real experiment ---
# You would run your chemical process with 'next_query' as input.
# Get the 'actual_yield_from_experiment'.
# Then, append this new input and its actual yield to your updated_inputs and updated_outputs arrays.
# Save the updated arrays to the .npy files.
# Repeat the process (load data, run gp_minimize for suggestions) in the next iteration.

# Example of how you would append the new data after your experiment:
# actual_yield_from_experiment = ... # Get this value from your real process/simulation
# updated_inputs = np.vstack([updated_inputs, next_query])
# updated_outputs = np.append(updated_outputs, actual_yield_from_experiment)
# np.save('/content/updated_inputs.npy', updated_inputs)
# np.save('/content/updated_outputs.npy', updated_outputs)
# print("\nAdded new data point from experiment.")

Best input parameters found so far: [0.49997, 0.500016, 0.499966, 0.499997]
Highest observed yield so far: 1879.7877797539552

Suggested next query to evaluate in your experiment:
[0.28772675700232375, 0.1805650161482346, 0.7456352721629655, 0.997732541850551]


In [ ]:
# Run the optimization using gp_minimize with LCB
res = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls,
    random_state=42,  # For reproducibility
    x0=updated_inputs.tolist() if updated_inputs.size > 0 else None,
    y0=[-o for o in updated_outputs.tolist()] if updated_outputs.size > 0 else None,
    n_initial_points=0, # Tell skopt not to generate random points if x0 is provided
    acq_func="LCB",     # Explicitly set the acquisition function to LCB (Lower Confidence Bound)
                        # LCB is used when minimizing the negative of the objective function to maximize the original objective
    kappa=1.96          # Adjust kappa as needed (common values range from 0.1 to 10)
                      # A common heuristic is to use the 95% confidence level (approx 1.96)
)

In [ ]:
# Cell 1: Install scikit-optimize
# Ensure this cell is run and completes successfully.
# You can choose a specific version or the latest.
!pip install scikit-optimize

# AFTER RUNNING THE ABOVE CELL, YOU MUST RESTART THE KERNEL.
# In Google Colab or Jupyter, you can usually do this via the 'Runtime' or 'Kernel' menu.

# Cell 2: Import modules and run code
# ONLY RUN THIS CELL AFTER RESTARTING THE KERNEL
import numpy as np
import matplotlib.pyplot as plt # Added matplotlib import as it's used later
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args
# Import Halton from skopt.sampler as it's used later in the notebook
from skopt.sampler import Halton
from scipy.optimize import minimize

# Define the search space for the four chemicals (assuming concentrations between 0 and 1)
space = [
    Real(0, 1, name='chemical_1'),
    Real(0, 1, name='chemical_2'),
    Real(0, 1, name='chemical_3'),
    Real(0, 1, name='chemical_4'),
]

# Load existing data
try:
    updated_inputs = np.load('/content/updated_inputs.npy')
    updated_outputs = np.load('/content/updated_outputs.npy')
except FileNotFoundError:
    updated_inputs = np.empty((0, 4))
    updated_outputs = np.empty((0,))
    print("Warning: updated_inputs.npy or updated_outputs.npy not found, starting with empty data.")

# Ensure updated_inputs and updated_outputs have the same length
min_len = min(updated_inputs.shape[0], updated_outputs.shape[0])
if updated_inputs.shape[0] != updated_outputs.shape[0]:
    print(f"Warning: Mismatch in number of samples between inputs ({updated_inputs.shape[0]}) and outputs ({updated_outputs.shape[0]}). Trimming data to {min_len} samples.")
updated_inputs = updated_inputs[:min_len]
updated_outputs = updated_outputs[:min_len]


# Define the objective function (replace this with your actual chemical process simulation or experiment)
@use_named_args(space)
def objective_function(**params):
    chemical_values = np.array([params['chemical_1'], params['chemical_2'],
                                params['chemical_3'], params['chemical_4']])

    # Check if this input combination has already been evaluated
    # Using np.isclose for floating point comparison
    if updated_inputs.size > 0 and any(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1)):
        existing_output_index = np.where(np.all(np.isclose(updated_inputs, chemical_values, rtol=1e-5, atol=1e-8), axis=1))[0][0]
        yield_value = updated_outputs[existing_output_index]
    else:
        # If not, you would run your chemical process here and get the 'yield_value'.
        # For demonstration, let's use a placeholder function that simulates a unimodal yield.
        # This example function has a maximum yield at (0.5, 0.5, 0.5, 0.5).
        # In a real scenario, this function would call your simulation or interact with your experiment.
        # yield_value = your_chemical_process_function(chemical_values)
        yield_value = 1500 * np.exp(-np.sum((chemical_values - 0.5)**2) / 0.1) + np.random.normal(0, 50) # Added noise

        # In a real scenario, you would NOT update your stored data here.
        # The data update should happen *after* you obtain the actual yield
        # from running the experiment with the suggested 'next_query'.
        # This objective function is only used by gp_minimize internally.


    # Since gp_minimize minimizes, we return the negative of the yield to maximize it.
    return -yield_value

# Determine the number of initial points from loaded data
n_initial_points_loaded = updated_inputs.shape[0] # Number of existing data points

# Set the total number of calls for gp_minimize
# If no data is loaded, we need initial random points. Set n_initial_points accordingly.
# If data is loaded, we rely on x0/y0 and set n_initial_points=0 for the call.
if n_initial_points_loaded == 0:
    n_initial_points_for_optimizer = 10 # Generate 10 random initial points if no data loaded
    n_calls = n_initial_points_for_optimizer + 5 # Total calls including the initial random ones and subsequent suggestions
    x0_value = None # No existing data to pass
    y0_value = None # No existing data to pass
else:
    n_initial_points_for_optimizer = 0 # Don't generate random points, use x0/y0
    n_calls = n_initial_points_loaded + 5 # Total calls including loaded data and subsequent suggestions
    x0_value = updated_inputs.tolist() # Pass existing inputs
    y0_value = [-o for o in updated_outputs.tolist()] # Pass negative of existing outputs


# Run the optimization using gp_minimize
res = gp_minimize(
    objective_function,
    space,
    n_calls=n_calls,
    random_state=42,  # For reproducibility
    x0=x0_value, # Provide existing inputs (or None if empty)
    y0=y0_value, # Provide negative of existing outputs (or None if empty)
    n_initial_points=n_initial_points_for_optimizer # Tell skopt how many random points to generate if no data is provided
)

# The 'res' object contains the results of the optimization.
# res.x is the best input found among all evaluated points
# res.fun is the minimum objective function value (negative of the maximum yield)

print("Best input parameters found so far:", res.x)
print("Highest observed yield so far:", -res.fun) # Negate to get the actual maximum yield

# To get the next recommended point to test in your actual chemical process:
# These are the points evaluated by gp_minimize beyond the initial points provided in x0.
# They are stored in res.x_iters after the first n_initial_points_loaded (if data was loaded)
# or after the n_initial_points_for_optimizer random points (if no data was loaded).
total_points_evaluated_internally = len(res.x_iters)

if total_points_evaluated_internally > n_initial_points_loaded:
    # The next query is the first point evaluated by the optimizer after the initial/loaded points
    next_query = res.x_iters[n_initial_points_loaded]
    print("\nSuggested next query to evaluate in your experiment:")
    print(next_query)

else:
    print("\nNo new queries were suggested by gp_minimize in this run.")
    print("This can happen if n_calls was not set high enough relative to the number of initial/loaded points.")
    print(f"Number of loaded points: {n_initial_points_loaded}")
    print(f"Total calls requested: {n_calls}")
    print(f"Total points evaluated by optimizer: {total_points_evaluated_internally}")


# --- After getting 'next_query', you would perform the real experiment ---
# You would run your chemical process with 'next_query' as input.
# Get the 'actual_yield_from_experiment'.
# Then, append this new input and its actual yield to your updated_inputs and updated_outputs arrays.
# Save the updated arrays to the .npy files.
# Repeat the process (load data, run gp_minimize for suggestions) in the next iteration.

# Example of how you would append the new data after your experiment:
# actual_yield_from_experiment = ... # Get this value from your real process/simulation
# updated_inputs = np.vstack([updated_inputs, next_query])
# updated_outputs = np.append(updated_outputs, actual_yield_from_experiment)
# np.save('/content/updated_inputs.npy', updated_inputs)
# np.save('/content/updated_outputs.npy', updated_outputs)
# print("\nAdded new data point from experiment.")

Best input parameters found so far: [0.49997, 0.500016, 0.499966, 0.499997]
Highest observed yield so far: 1879.7877797539552

Suggested next query to evaluate in your experiment:
[0.28772675700232375, 0.1805650161482346, 0.7456352721629655, 0.997732541850551]
